In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2, poisson, norm
from math import erfc, sqrt, log, exp, lgamma
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

from src.on import (
    poisson_tail_on,
    r_stat_on,
    q_stat_on,
    norm_survival,
    r_star_on,
    pvals_on,
    expected_significance_on,
)


In [ ]:
# -------- Simple counting: p-value grids --------
b_vec   = np.array([0, 1, 2, 5])
s0_vec  = np.array([0.1, 0.5, 1.0, 2.0, 5.0, 10.0])

out_dir  = Path("plots")
pdf_name = "simple_pval.pdf"

out_dir.mkdir(parents=True, exist_ok=True)
pdf_path = out_dir / pdf_name

with PdfPages(pdf_path) as pdf:
    for b_fixed in b_vec:
        nrows = len(s0_vec)
        fig_height = 3.8 * nrows + 1.0
        fig, axes = plt.subplots(
            nrows=nrows, ncols=1,
            figsize=(14, fig_height),
            sharex=False
        )

        if nrows == 1:
            axes = [axes]

        for ax, s0 in zip(axes, s0_vec):
            mu0 = float(s0) + float(b_fixed)
            n_vals = np.arange(0, int(np.ceil(mu0 + 5.0 * np.sqrt(mu0))) + 1, dtype=int)
            out = pvals_on(float(s0), float(b_fixed), n_vals)
            p_true = out['p_true']
            p_r = out['p_r']
            p_rstar = out['p_rstar']

            ax.plot(n_vals, p_true,  marker="x", linestyle="None", label="MC", color="tab:green")
            ax.plot(n_vals, p_r,     marker="o", linestyle="None", label="1 − Φ(r)", color="tab:blue")
            ax.plot(n_vals, p_rstar, marker="p", linestyle="None", label="1 − Φ(r*)", color="tab:orange")

            ax.set_yscale("log")
            ax.set_ylabel("p-value", fontsize=12)

            mu0 = s0 + b_fixed
            ax.set_title(f"s₀ = {s0},  b = {b_fixed},  μ₀ = s₀ + b = {mu0}", fontsize=13)

            ax.grid(True, which="both", linestyle=":", alpha=0.5)
            ax.tick_params(axis="both", labelsize=11)

        axes[-1].set_xlabel("Observed count n", fontsize=12)
        axes[0].legend(fontsize=11)

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved all plots to: {pdf_path.resolve()}")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2, poisson, norm
from math import erfc, sqrt, log, exp, lgamma
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

from src.on import (
    poisson_tail_on,
    r_stat_on,
    q_stat_on,
    norm_survival,
    r_star_on,
    pvals_on,
    expected_significance_on,
)

# -------- Simple counting: Asimov vs median expected Z (per-s page) --------
s_vec = np.array([0.5, 0.6, 0.8, 1.0, 2.0, 5.0, 10])
b_values = np.logspace(-1, 2, 200)  # 0.1 to 100

out_dir = Path("plots")
out_dir.mkdir(exist_ok=True)
pdf_path = out_dir / "simple_medsig.pdf"

with PdfPages(pdf_path) as pdf:
    for s_true in s_vec:
        r_list = np.vectorize(lambda bb: r_stat_on_on(s0=0.0, b=bb, n=s_true + bb))(b_values)
        rstar_list = np.vectorize(lambda bb: r_star_on_on(s0=0.0, b=bb, n=s_true + bb))(b_values)
        zmed = expected_significance_on(s_true, b_values)["Z_mc_median"]

        fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
        ax.plot(b_values, r_list, label=fr"Asimov r, s_true={s_true}")
        ax.plot(b_values, rstar_list, linestyle="--", label=fr"Asimov r*, s_true={s_true}")
        ax.plot(b_values, zmed, linestyle="None", marker="x", label=fr"MC median Z, s_true={s_true}")

        ax.set_xscale("log")
        ax.set_xlabel("b")
        ax.set_ylabel("Z")
        ax.set_ylim(bottom=-1)
        ax.grid(True, which="both", ls="--", alpha=0.35)
        ax.legend(fontsize=9)
        ax.set_title(fr"Asimov vs MC median Z, $s_{{\mathrm{{true}}}} = {s_true}$")

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved all plots to: {pdf_path.resolve()}")
